In [4]:
# Install dotenv only if running in Colab
try:
    import google.colab
    %pip install dotenv -q
except ImportError:
    pass  # Running locally, dotenv should already be installed


In [5]:
import google.generativeai as genai
import os

# --- Detect environment and load API keys ---
try:
    from google.colab import userdata
    IN_COLAB = True
    print('🌐 Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('💻 Running locally')

if IN_COLAB:
    # Load keys from Colab Secrets
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    try:
        os.environ['serp_api_key'] = userdata.get('serp_api_key')
    except Exception:
        print('⚠️ serp_api_key not found in Colab Secrets, skipping.')
else:
    # Load keys from .env file
    from dotenv import load_dotenv, find_dotenv
    env_file = find_dotenv()
    if not env_file and os.path.exists('.env'):
        env_file = '.env'
    if env_file:
        load_dotenv(env_file)
        print(f'✅ Loaded .env from: {env_file}')
    else:
        print('⚠️ Warning: Could not find .env file.')
    GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')

# --- Validate and configure ---
if not GOOGLE_API_KEY:
    raise ValueError('❌ GOOGLE_API_KEY not found. '
                     'In Colab: add it to Secrets. '
                     'Locally: add it to your .env file.')

genai.configure(api_key=GOOGLE_API_KEY)
print('✅ Gemini API configured successfully.')


🌐 Running in Google Colab


TimeoutException: Requesting secret GOOGLE_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

# Plan for podcast generation
1. Planner model(should have basic idea about the topic)
2. RAG Search (Retrieval-Augmented Generation)
3. Eliminate Factual Errors (Post-RAG)
4. Script Generation
5. Factual Error Check (again)
6. Text to speech.


# 1. Planner Model

In [9]:
from typing import List, Dict
import os
import google.generativeai as genai
import json
import re

Segment = Dict[str, object]

def plan_podcast(user_input: str) -> List[Segment]:
    # Configure the Gemini API
    GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model = genai.GenerativeModel('gemini-2.5-flash') # Using a suitable model for planning

    # Create a prompt for the Gemini model to generate the podcast plan
    prompt = f"""
    You are a podcast planner. Your task is to create a detailed outline for a podcast episode based on the user's input.
    For each segment of the podcast, provide a title, a brief description, and a list of search queries that can be used
    to retrieve relevant information for that segment. We will use this information to retreive relavent information from
    the internet.

    User input: {user_input}

    Please structure your response as a list of dictionaries, where each dictionary represents a segment with the following keys:
    "segment_id": An integer representing the order of the segment (starting from 1).
    "segment_title": A concise title for the segment.
    "segment_description": A brief description of what the segment will cover.
    "queries": A list of strings, where each string is a search query for retrieving information related to the segment. There should be atmost 2 queries per segment.

    Example for "Space Exploration News":
    [
      {{ "segment_id": 1, "segment_title": "Mars Missions", "segment_description": "Updates on current and future missions to Mars.", "queries": ["Mars mission news","Mars current news"] }},
      {{ "segment_id": 2, "segment_title": "New Discoveries", "segment_description": "Recent findings in astronomy and space science.", "queries": ["new space discoveries"] }},
      {{ "segment_id": 3, "segment_title": "Space Tourism", "segment_description": "The latest on commercial space travel.", "queries": ["space tourism news"] }},
    ]

    Please generate the plan for "{user_input}".
    """

    # Generate the plan using the Gemini model
    response = gemini_model.generate_content(prompt)
    cleared_response = re.sub(r"^```json|```$", "", response.text.strip())

    # Parse the model's response (assuming it returns a string that can be evaluated as a list of dictionaries)
    # It's important to handle potential errors in parsing the model's output.
    try:
        planned_segments = eval(cleared_response)
    except Exception as e:
        print(f"Error parsing model response: {e}")
        print("Model response text:")
        print(response.text)
        planned_segments = [] # Return an empty list or handle the error as appropriate

    return planned_segments

# The rest of the code in this cell will remain the same or be moved to another cell if it's for execution.
# For now, keeping the function definition here.
planned_segments=[] # This line was part of the original cell, keeping it here for now.


Output structure from planner model.

```
[
  {
    "segment_id": 1,
    "segment_title": "...",
    "segment_description": "...",
    "queries": ["q1", "q2"]
  },
  ...
]
```



In [10]:
user_input = input("enter a topic : ")
planner_output = plan_podcast(user_input)
print(planner_output)
print("Generated Plan:")
for segment in planner_output:
    print(f"Segment ID: {segment['segment_id']}")
    print(f"Segment Title: {segment['segment_title']}")
    print(f"Segment Description: {segment['segment_description']}")
    print(f"Queries: {segment['queries']}")
    print("-" * 20)

[{'segment_id': 1, 'segment_title': "Introducing 'Sarvam Maya'", 'segment_description': "An introduction to the Malayalam movie 'Sarvam Maya', its release, and initial context within the industry.", 'queries': ['Sarvam Maya Malayalam movie release date', 'Sarvam Maya movie initial buzz']}, {'segment_id': 2, 'segment_title': 'Plot and Premise (Spoiler-Free)', 'segment_description': "A brief, spoiler-free overview of the film's core plot, genre, and what viewers can expect without revealing twists.", 'queries': ['Sarvam Maya Malayalam movie plot summary', 'Sarvam Maya movie genre']}, {'segment_id': 3, 'segment_title': 'Cast, Crew & Performances', 'segment_description': 'Discussion about the director, writers, key actors, and notable performances that shaped the film.', 'queries': ['Sarvam Maya Malayalam movie cast and crew', 'Sarvam Maya movie director']}, {'segment_id': 4, 'segment_title': 'Themes & Cinematic Elements', 'segment_description': "A deeper dive into the movie's underlying t

In [ ]:
for segment in planner_output:
    print(f"Segment ID: {segment['segment_id']}")
    print(f"Segment Title: {segment['segment_title']}")
    print(f"Segment Description: {segment['segment_description']}")
    print(f"Queries: {segment['queries']}")
    print("-" * 20)

Segment ID: 1
Segment Title: Introducing 'Sarvam Maya'
Segment Description: An introduction to the Malayalam movie 'Sarvam Maya', its release, and initial context within the industry.
Queries: ['Sarvam Maya Malayalam movie release date', 'Sarvam Maya movie initial buzz']
--------------------
Segment ID: 2
Segment Title: Plot and Premise (Spoiler-Free)
Segment Description: A brief, spoiler-free overview of the film's core plot, genre, and what viewers can expect without revealing twists.
Queries: ['Sarvam Maya Malayalam movie plot summary', 'Sarvam Maya movie genre']
--------------------
Segment ID: 3
Segment Title: Cast, Crew & Performances
Segment Description: Discussion about the director, writers, key actors, and notable performances that shaped the film.
Queries: ['Sarvam Maya Malayalam movie cast and crew', 'Sarvam Maya movie director']
--------------------
Segment ID: 4
Segment Title: Themes & Cinematic Elements
Segment Description: A deeper dive into the movie's underlying theme

# 2. Searching internet (Retrieval)

## New Implementation

In [ ]:
# %pip install google-search-results
# %pip install serpapi
!python.exe -m pip install --upgrade pip


  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2


In [ ]:
# planner_output = [
#     {
#         "segment_id": 1,
#         "segment_title": "Introducing 'Bha Bha Bha'",
#         "segment_description": "Intro segment",
#         "queries": [
#             "Bha Bha Bha Malayalam movie release date",
#             "Bha Bha Bha movie genre"
#         ]
#     },
#     {
#         "segment_id": 2,
#         "segment_title": "The Minds Behind 'Bha Bha Bha'",
#         "segment_description": "Cast and crew",
#         "queries": [
#             "Bha Bha Bha Malayalam movie cast and crew",
#             "Bha Bha Bha Malayalam movie plot overview"
#         ]
#     }
# ]


In [ ]:
from serpapi import Client
# from google.colab import userdata

# SERPAPI_KEY = "YOUR_SERPAPI_KEY"
SERPAPI_KEY = os.getenv("serp_api_key")
client = Client(api_key=SERPAPI_KEY)


def search_serpapi(query, num_results=5):
    """
    Executes a single SerpAPI Google search
    Returns a list of dicts with title, link, snippet
    """
    results = client.search({
        "engine": "google",
        "q": query,
        "num": num_results,
        "hl": "en",
        "gl": "in"
    })

    organic = results.get("organic_results", [])

    cleaned_results = []
    for r in organic:
        cleaned_results.append({
            "title": r.get("title"),
            "url": r.get("link"),
            "snippet": r.get("snippet")
        })

    return cleaned_results


: 

In [ ]:
def collect_segment_sources(planner_output, max_queries_per_segment=2, results_per_query=5):
    """
    Takes planner output and returns segment-wise sources
    """
    segment_sources = []
    global_seen_urls = set()

    for segment in planner_output:
        segment_result = {
            "segment_id": segment["segment_id"],
            "segment_title": segment["segment_title"],
            "sources": []
        }

        queries = segment.get("queries", [])[:max_queries_per_segment]

        for query in queries:
            search_results = search_serpapi(query, num_results=results_per_query)

            for r in search_results:
                url = r["url"]

                # Global URL deduplication
                if not url or url in global_seen_urls:
                    continue

                global_seen_urls.add(url)

                segment_result["sources"].append({
                    "url": url,
                    "title": r["title"],
                    "snippet": r["snippet"],
                    "source_query": query
                })

        segment_sources.append(segment_result)

    return segment_sources


In [ ]:
from urllib.parse import urlparse

BLOCKED_DOMAINS = {
    "youtube.com",
    "youtu.be",
    "instagram.com",
    "twitter.com",
    "x.com",
    "facebook.com"
}

LOW_VALUE_KEYWORDS = [
    "/tickets",
    "/book",
    "/buy"
]

MAX_SOURCES_PER_SEGMENT = 4


def filter_segment_sources(sources):
    filtered = []
    seen_domains = set()

    for src in sources:
        url = src["url"]
        domain = urlparse(url).netloc.lower()

        # Rule 1: block bad domains
        if any(bad in domain for bad in BLOCKED_DOMAINS):
            continue

        # Rule 2: one source per domain
        if domain in seen_domains:
            continue

        # Rule 3: deprioritize low-value URLs
        if any(keyword in url.lower() for keyword in LOW_VALUE_KEYWORDS):
            continue

        seen_domains.add(domain)
        filtered.append(src)

        # Rule 4: cap total sources
        if len(filtered) >= MAX_SOURCES_PER_SEGMENT:
            break

    return filtered


In [ ]:
segment_sources = collect_segment_sources(planner_output)

for segment in segment_sources:
    segment["sources"] = filter_segment_sources(segment["sources"])

from pprint import pprint
pprint(segment_sources)


[{'segment_id': 1,
  'segment_title': 'Intro: The AI Model Showdown',
  'sources': [{'snippet': 'Coding: Claude is the best, but Gemini is the most '
                          'cost effective · Gemini 2.5 Pro and Flash are solid '
                          'models for the price. · Jules is an async ...',
               'source_query': 'Gemini vs other AI models overview',
               'title': 'ChatGPT vs Claude vs Gemini: The Best AI Model for '
                        'Each ...',
               'url': 'https://creatoreconomy.so/p/chatgpt-vs-claude-vs-gemini-the-best-ai-model-for-each-use-case-2025'},
              {'snippet': 'Gemini is better at multi-turn conversations, '
                          'making it more effective at leveraging data from '
                          'different modalities. It tends to be more concise '
                          '...',
               'source_query': 'Gemini vs other AI models overview',
               'title': 'Gemini vs ChatGPT: Comparing 

## Old Implementation

In [ ]:
%pip install trafilatura
%pip install serpapi
%pip install pinecone
%pip install google-generativeai serpapi trafilatura faiss-cpu sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.9/837.9 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.5/315.5 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.7/274.7 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.9/745.9 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 96.6 MB/s eta 0:00:00


In [ ]:
from serpapi import Client

from trafilatura import fetch_url, extract
from openai import OpenAI
import pinecone

In [ ]:
import google.generativeai as genai
from serpapi import Client
import trafilatura
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import time
from google.colab import userdata

# ====== CONFIG ======
SERP_API_KEY = userdata.get("serp_api")  # or use Brave Search API if preferred
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
NUM_RESULTS = 5
NUM_CONTEXTS = 3
CHUNK_SIZE = 500

# ====== Setup ======
genai.configure(api_key=GOOGLE_API_KEY)
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def search_web(query):
    search = Client()
    results = search.search(q=query, api_key=SERP_API_KEY).get("organic_results", [])
    return [r.get("link") for r in results[:NUM_RESULTS]]

def extract_text(url):
    try:
        downloaded = trafilatura.fetch_url(url)
        if downloaded:
            return trafilatura.extract(downloaded)
    except Exception as e:
        print(f"Error extracting text from {url}: {e}")
    return None

def chunk_text(text, chunk_size=CHUNK_SIZE):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

def get_embedding(text):
    return np.array(embedder.encode(text), dtype="float32")

def retrieve_context(segments):
    all_context = {}
    vector_dim = len(get_embedding("test"))
    index = faiss.IndexFlatL2(vector_dim)
    stored_chunks, stored_meta = [], []

    for seg in segments:
        seg_id = seg["segment_id"]
        for query in seg["queries"]:
            urls = search_web(query)
            time.sleep(3)
            for url in urls:
                text = extract_text(url)
                if text:
                    chunks = chunk_text(text)
                    for chunk in chunks:
                        emb = get_embedding(chunk)
                        index.add(np.array([emb]))
                        stored_chunks.append(chunk)
                        stored_meta.append({"segment_id": seg_id, "source": url})

    for seg in segments:
        seg_id = seg["segment_id"]
        combined_queries = " ".join(seg["queries"])
        emb = get_embedding(combined_queries)
        D, I = index.search(np.array([emb]), NUM_CONTEXTS)
        top_chunks = [stored_chunks[i] for i in I[0]]
        all_context[seg_id] = top_chunks

    return all_context

# ====== Example Run ======
segments_json = [
    {
    "segment_id": 1,
    "segment_title": "Unveiling Antigravity: Google's Agentic Development Tool",
    "segment_description": "An introduction to Google's new 'Antigravity' tool. What it is, its purpose, and why it's a significant development in the world of AI and software engineering.",
    "queries": [
      "Google Antigravity tool announcement",
      "what is Google Antigravity",
      "Google agentic development tool Antigravity explained",
      "Antigravity purpose Google"
    ]
  },
  {
    "segment_id": 2,
    "segment_title": "Decoding Agentic AI: The Foundation of Antigravity",
    "segment_description": "A deeper dive into what 'agentic development' and 'AI agents' mean. Discuss the core concepts, their growing importance, and how they differ from traditional AI models.",
    "queries": [
      "what is agentic AI development",
      "AI agents explained",
      "importance of AI agent frameworks",
      "agentic computing paradigm",
      "Google's approach to AI agents"
    ]
  },
  {
    "segment_id": 3,
    "segment_title": "Inside Antigravity: How It Works and Key Features",
    "segment_description": "Explore the technical aspects of Antigravity. Discuss its core features, functionality, how developers interact with it, the types of tasks it streamlines, and its underlying architecture.",
    "queries": [
      "Google Antigravity features",
      "how does Google Antigravity work",
      "Antigravity technical details",
      "Antigravity architecture Google",
      "developer workflow with Google Antigravity"
    ]
  }
]

context_data = retrieve_context(segments_json)

# ====== Use Gemini to write script ======
for seg in segments_json:
    seg_id = seg["segment_id"]
    context = "\n".join(context_data[seg_id])
    # prompt = f"""
    # You are a podcast scriptwriter.
    # Segment Title: {seg['segment_title']}
    # Description: {seg['segment_description']}

    # Context information:
    # {context}

    # Write a conversational, engaging script for this segment.
    # """
    print("context", context)
    # response = genai.GenerativeModel('gemini-1.5-flash-latest').generate_content(prompt)
    # print(f"\n--- Segment {seg_id} Script ---\n")
    # print(response.text)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ERROR:trafilatura.downloads:not a 200 response: 403 for URL https://www.science.org/content/article/house-panel-concludes-covid-19-pandemic-came-lab-leak
ERROR:trafilatura.downloads:download error: https://www.nature.com/articles/d41586-024-03026-9 HTTPSConnectionPool(host='idp.nature.com', port=443): Max retries exceeded with url: https://idp.nature.com/transit?redirect_uri=https%3A%2F%2Fwww.nature.com%2Farticles%2Fd41586-024-03026-9&code=f251aad8-4bd5-413a-bd67-80471873fcd7 (Caused by ResponseError('too many redirects'))


context Timeline of the COVID-19 pandemic | Part of a series on the | | COVID-19 pandemic | |---| | | COVID-19 portal | The timeline of the COVID-19 pandemic lists the articles containing the chronology and epidemiology of SARS-CoV-2,[1] the virus that causes the coronavirus disease 2019 (COVID-19) and is responsible for the COVID-19 pandemic. The first human cases of COVID-19 occurred in Wuhan, People's Republic of China, on or about 17 November 2019.[2] The first confirmed human case in the United States was on 19 January 2020. The World Health Organization declared the COVID-19 outbreak a Public Health Emergency of International Concern (PHEIC) on 30 January 2020, and first referred to it as a pandemic on 11 March 2020.[3][4] The WHO ended the PHEIC on 5 May 2023.[5] Worldwide timelines by month and year [edit]The 2019 and January 2020 timeline articles include the initial responses as subsections, and more comprehensive timelines by nation-state are listed below this section. The f

In [ ]:
import os
import json
import requests
import faiss
import numpy as np
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer

# ====== CONFIG ======
SERP_API_KEY = userdata.get("serp_api")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

MAX_RESULTS_PER_QUERY = 10
MAX_CHUNKS_PER_SEGMENT = 15

# ====== HELPERS ======
def serp_search(query):
    url = "https://serpapi.com/search.json"
    params = {
        "q": query,
        "hl": "en",
        "gl": "us",
        "api_key": SERP_API_KEY
    }
    r = requests.get(url, params=params)
    r.raise_for_status()
    data = r.json()
    return data.get("organic_results", [])

def fetch_clean_text(url):
    try:
        resp = requests.get(url, timeout=10)
        soup = BeautifulSoup(resp.text, "html.parser")

        for script in soup(["script", "style"]):
            script.decompose()

        text = " ".join(soup.stripped_strings)
        return text if len(text) > 200 else None
    except:
        return None

def chunk_text(text, max_words=200):
    words = text.split()
    return [" ".join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

# ====== MAIN RAG PIPELINE ======
def build_rag_context(segments):
    rag_store = []

    for seg in segments:
        all_chunks = []

        for query in seg["queries"]:
            results = serp_search(query)

            for res in results[:MAX_RESULTS_PER_QUERY]:
                url = res.get("link")
                snippet = res.get("snippet", "")

                page_text = fetch_clean_text(url)
                if not page_text:
                    continue

                chunks = chunk_text(page_text)
                for c in chunks:
                    all_chunks.append({
                        "content": c,
                        "source_url": url,
                        "search_snippet": snippet
                    })

        all_chunks = all_chunks[:MAX_CHUNKS_PER_SEGMENT]

        embeddings = embed_model.encode([c["content"] for c in all_chunks])
        index = faiss.IndexFlatL2(embeddings.shape[1])
        index.add(np.array(embeddings))

        rag_store.append({
            "segment_id": seg["segment_id"],
            "segment_title": seg["segment_title"],
            "segment_description": seg["segment_description"],
            "chunks": all_chunks,
            "faiss_index": index
        })

    return rag_store

# ====== Example usage ======
segments_json = [
    {
        "segment_id": 1,
        "segment_title": "Origins and Spread of COVID-19",
        "segment_description": "Tracing the initial outbreak...",
        "queries": ["COVID-19 origins Wuhan", "COVID-19 global spread timeline"]
    }
]
rag_data = build_rag_context(segments_json)
json.dump(rag_data, open("rag_store.json", "w"), indent=2)


[{'segment_id': 1, 'segment_title': 'Origins and Spread of COVID-19', 'segment_description': 'Tracing the initial outbreak...', 'chunks': [{'content': '%PDF-1.5 %���� 1 0 obj < >>> endobj 2 0 obj < > endobj 3 0 obj < >/ExtGState< >/Font< >/ProcSet[/PDF/Text/ImageB/ImageC/ImageI] >>/MediaBox[ 0 0 612 792] /Contents 4 0 R/Group< >/Tabs/S/StructParents 0>> endobj 4 0 obj < > stream x��\\Ys�F\x12~w���<�)\x11����u�J��w7���+\x0f�H��H�\x02�b�ﷻ\x07\x00A\x01#���lR�I\x02��s��=��\'������ � ��\\�{\x7f)ξ�\x14����� \'o� D\x14���D�z�})\x17\'o>\x7f#��7�ч��M���ɛ����v\x11��F�|�F|�=<����evS#�E]g7��\\|9�2���]}�_�����EV�8�����\x7f,���l�\x03�|���R�\x10\x18^�T�hzw\x05� �� W˓7��BE�KB\x11\x07�\x17%� ���-���\'��h}� P�<�\x0f�X\\�|����\x7f_|�����o�O\x7f\x13W�<y�-\x00�L��:Jt�z* PB�_&�(/ E��K�Cb���aZ�����\x16�S/�D�$��<��ޙ�6\x1b���`LͶ�`h2\x08 �\x08�M9v�(�\x11&�E��dd\x05F�\x02$\x0e��\x03�\x02�\x1b\x01\x03�z?�K\x18��"F�$�|���Ǔ\x17+�\x07�\x16��\'�e�\x01�b�����|K�\'Qs�\x18%^x \x12\x03.\x12�ċ�R\x117�D� �%#%�� �&K%^z \x12

TypeError: Object of type IndexFlatL2 is not JSON serializable

# Content fetching & clean text extraction

In [ ]:
%pip install trafilatura requests


In [ ]:
import requests
import trafilatura
from time import sleep


def fetch_and_clean_text(url, timeout=10):
    """
    Fetches a URL and extracts clean main article text.
    Returns cleaned text or None if extraction fails.
    """
    try:
        response = requests.get(
            url,
            timeout=timeout,
            headers={
                "User-Agent": "Mozilla/5.0 (compatible; PodcastBot/1.0)"
            }
        )

        if response.status_code != 200:
            return None

        downloaded = trafilatura.extract(
            response.text,
            include_comments=False,
            include_tables=False
        )

        if downloaded and len(downloaded.split()) > 100:
            return downloaded.strip()

        return None

    except Exception:
        return None


In [ ]:
def enrich_segments_with_content(segment_sources, delay=1.0):
    """
    For each source URL, fetch and attach clean article text.
    """
    for segment in segment_sources:
        for source in segment["sources"]:
            url = source["url"]

            clean_text = fetch_and_clean_text(url)
            source["clean_text"] = clean_text

            # polite crawling
            sleep(delay)

    return segment_sources


In [ ]:
def drop_empty_sources(segment_sources):
    for segment in segment_sources:
        segment["sources"] = [
            s for s in segment["sources"]
            if s.get("clean_text")
        ]
    return segment_sources


In [ ]:
def tag_source_type(source):
    url = source["url"]

    if "imdb.com" in url:
        source["content_type"] = "metadata"
    else:
        source["content_type"] = "article"

    return source


In [ ]:
segment_sources = enrich_segments_with_content(segment_sources)
segment_sources = drop_empty_sources(segment_sources)
for segment in segment_sources:
    for source in segment["sources"]:
        tag_source_type(source)

from pprint import pprint
pprint(segment_sources)


[{'segment_id': 1,
  'segment_title': 'Intro: The AI Model Showdown',
  'sources': [{'clean_text': 'ChatGPT vs Claude vs Gemini: The Best AI Model '
                             'for Each Use Case in 2025\n'
                             'Comparing all 3 AI models for coding, writing, '
                             'multimodal, and 6 other use cases\n'
                             'Dear subscribers,\n'
                             'Today, I want to share an updated guide on the '
                             'best AI models by use case.\n'
                             'I made a video testing Claude 4, ChatGPT O3, and '
                             'Gemini 2.5 head-to-head for coding, writing, '
                             'deep research, multimodal and more. What I found '
                             'was that the "best" model depends on what '
                             "you're trying to do. Watch me test all 3 live "
                             'here:\n'
                         

# Chunking article-type sources into LLM-friendly chunks

In [ ]:
import re

def clean_text_for_chunking(text):
    # Remove [edit], [citation needed], etc.
    text = re.sub(r"\[.*?\]", "", text)

    # Remove multiple newlines
    text = re.sub(r"\n{2,}", "\n\n", text)

    # Remove trailing references section
    text = re.split(r"\nReferences|\nExternal links", text)[0]


    return text.strip()

In [ ]:
def trim_wikipedia_for_intro(text):
    STOP_SECTIONS = [
        "\nPlot",
        "\nCast",
        "\nProduction",
        "\nBox office",
        "\nControversy"
    ]

    for stop in STOP_SECTIONS:
        if stop in text:
            text = text.split(stop)[0]

    return text.strip()


In [ ]:
def chunk_text(text, min_words=150, max_words=350):
    paragraphs = [p.strip() for p in text.split("\n") if p.strip()]

    chunks = []
    current_chunk = []
    current_word_count = 0

    for para in paragraphs:
        words = para.split()
        para_len = len(words)

        # If paragraph itself is huge, force split
        if para_len > max_words:
            for i in range(0, para_len, max_words):
                chunk_words = words[i:i+max_words]
                chunks.append(" ".join(chunk_words))
            continue

        if current_word_count + para_len <= max_words:
            current_chunk.append(para)
            current_word_count += para_len
        else:
            if current_word_count >= min_words:
                chunks.append("\n".join(current_chunk))
                current_chunk = [para]
                current_word_count = para_len
            else:
                current_chunk.append(para)
                current_word_count += para_len

    if current_chunk:
        chunks.append("\n".join(current_chunk))

    return chunks


In [ ]:
def build_segment_chunks(segment_sources):
    all_segment_chunks = []

    for segment in segment_sources:
        segment_chunk_obj = {
            "segment_id": segment["segment_id"],
            "segment_title": segment["segment_title"],
            "chunks": []
        }

        chunk_counter = 1

        for source in segment["sources"]:
            if source.get("content_type") != "article":
                continue

            text = source.get("clean_text")
            if not text:
                continue

            cleaned_text = clean_text_for_chunking(text)
            if "wikipedia.org" in source["url"] and segment["segment_id"] == 1:
              cleaned_text = trim_wikipedia_for_intro(cleaned_text)
            chunks = chunk_text(cleaned_text)

            for chunk in chunks:
                segment_chunk_obj["chunks"].append({
                    "chunk_id": f"{segment['segment_id']}_{chunk_counter}",
                    "source_url": source["url"],
                    "text": chunk
                })
                chunk_counter += 1

        all_segment_chunks.append(segment_chunk_obj)

    return all_segment_chunks


In [ ]:
segment_chunks = build_segment_chunks(segment_sources)

from pprint import pprint
pprint(segment_chunks)


[{'chunks': [{'chunk_id': '1_1',
              'source_url': 'https://creatoreconomy.so/p/chatgpt-vs-claude-vs-gemini-the-best-ai-model-for-each-use-case-2025',
              'text': 'ChatGPT vs Claude vs Gemini: The Best AI Model for Each '
                      'Use Case in 2025\n'
                      'Comparing all 3 AI models for coding, writing, '
                      'multimodal, and 6 other use cases\n'
                      'Dear subscribers,\n'
                      'Today, I want to share an updated guide on the best AI '
                      'models by use case.\n'
                      'I made a video testing Claude 4, ChatGPT O3, and Gemini '
                      '2.5 head-to-head for coding, writing, deep research, '
                      'multimodal and more. What I found was that the "best" '
                      "model depends on what you're trying to do. Watch me "
                      'test all 3 live here:\n'
                      'Watch now on YouTube.\n'
  

# Multi-speaker script generation with Gemini

In [ ]:
from google.colab import userdata
import google.generativeai as genai

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
genai.configure(api_key=GOOGLE_API_KEY)

gemini_model = genai.GenerativeModel('gemini-2.5-flash') # Using a suitable model for planning

# gemini_model = genai.GenerativeModel(
#     model_name="gemini-2.5-flash",
#     generation_config={
#         "temperature": 0.7,
#     }
# )


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
def build_multi_speaker_prompt(segment):
    source_text = ""
    for i, chunk in enumerate(segment["chunks"], start=1):
        source_text += f"\n[Source {i}]\n{chunk['text']}\n"

    prompt = f"""
You are a professional podcast script writer.

You are writing ONE FULL podcast segment.

STRICT RULES:
- Use ONLY the information from the SOURCE MATERIAL.
- Do NOT invent facts.
- Do NOT add information not present in the sources.
- Write a NATURAL, conversational script.
- Avoid sounding like a summary or article.
- Avoid spoilers unless unavoidable.

SEGMENT TITLE:
{segment['segment_title']}

SEGMENT REQUIREMENTS:
- Write at least 12 dialogue turns.
- Both speakers must speak multiple times.
- The HOST should explain facts and context.
- The CO-HOST should react, question, or comment.
- Reference information from MULTIPLE sources.
- Do NOT stop early.

SPEAKERS:
HOST – guides the discussion and provides context
CO-HOST – reacts, adds opinions, and challenges ideas

SOURCE MATERIAL (use facts ONLY from here):
{source_text}

OUTPUT FORMAT (STRICT):
HOST: ...
CO-HOST: ...
HOST: ...
CO-HOST: ...
"""

    return prompt.strip()


In [ ]:
def generate_segment_script(segment):
    """
    Generates a multi-speaker podcast script for one segment.
    """

    prompt = build_multi_speaker_prompt(segment)

    response = gemini_model.generate_content(prompt)

    script_text = response.text.strip()

    return {
        "segment_id": segment["segment_id"],
        "segment_title": segment["segment_title"],
        "script": script_text
    }


In [ ]:
def generate_full_podcast_script(segment_chunks):
    """
    Generates scripts for all segments.
    """

    full_script = []

    for segment in segment_chunks:
        print(f"Generating script for segment {segment['segment_id']}...")
        segment_script = generate_segment_script(segment)
        full_script.append(segment_script)

    return full_script


In [ ]:
# segment_chunks comes from your chunking step
podcast_scripts = generate_full_podcast_script(segment_chunks)
from pprint import pprint
pprint(podcast_scripts)


Generating script for segment 1...
Generating script for segment 2...
Generating script for segment 3...
Generating script for segment 4...
Generating script for segment 5...
Generating script for segment 6...
Generating script for segment 7...
Generating script for segment 8...
Generating script for segment 9...
Generating script for segment 10...
Generating script for segment 11...
[{'script': "HOST: Welcome back to the show! Today, we're diving deep into a "
            "topic that's been heating up the tech world: the battle of the AI "
            'models. Ever since OpenAI launched ChatGPT in late 2022, we’ve '
            'seen a massive race among tech giants to develop and refine their '
            'own AI tools.\n'
            '\n'
            'CO-HOST: It really feels like a new AI assistant pops up every '
            'other week! It can be overwhelming trying to figure out which one '
            'to even try, let alone which one is "the best."\n'
            '\n'
       

# Script Clean Up


In [ ]:
import re

def strip_source_markers(text: str) -> str:
    """
    Removes (Source X), (Source 1, Source 2), (Note 1), etc.
    Designed specifically for TTS cleanup.
    """

    # Remove parenthesized Source/Note references
    text = re.sub(
        r"\(\s*(Source|Sources|Note|Notes)[^)]*\)",
        "",
        text,
        flags=re.IGNORECASE
    )

    # Remove leftover extra spaces
    text = re.sub(r"\s{2,}", " ", text)

    # Clean up spaces before punctuation
    text = re.sub(r"\s+([.,!?])", r"\1", text)

    return text.strip()


In [ ]:
def cleanup_scripts_for_tts(podcast_scripts):
    for segment in podcast_scripts:
        segment["script"] = strip_source_markers(segment["script"])
    return podcast_scripts


In [ ]:
podcast_scripts = cleanup_scripts_for_tts(podcast_scripts)


In [ ]:
def normalize_newlines(text: str) -> str:
    return re.sub(r"\n{3,}", "\n\n", text)


In [ ]:
def add_tts_pauses(text: str) -> str:
    text = re.sub(r"HOST:", "HOST: <break time='300ms'/>", text)
    text = re.sub(r"CO-HOST:", "CO-HOST: <break time='300ms'/>", text)
    return text


In [ ]:
for segment in podcast_scripts:
    script = segment["script"]
    script = strip_source_markers(script)
    script = normalize_newlines(script)
    # script = add_tts_pauses(script)  # optional
    segment["script"] = script


In [ ]:
pprint(podcast_scripts)
print(podcast_scripts)


[{'script': "HOST: Welcome back to the show! Today, we're diving deep into a "
            "topic that's been heating up the tech world: the battle of the AI "
            'models. Ever since OpenAI launched ChatGPT in late 2022, we’ve '
            'seen a massive race among tech giants to develop and refine their '
            'own AI tools. CO-HOST: It really feels like a new AI assistant '
            'pops up every other week! It can be overwhelming trying to figure '
            'out which one to even try, let alone which one is "the best." '
            "HOST: You've hit on the core challenge, because as it turns out, "
            'there isn\'t one single "best" model. What\'s clear from recent '
            'comparisons is that the ideal AI model really depends on what '
            "you're trying to achieve. We're talking about ChatGPT, Claude, "
            "and Gemini as the major players here. CO-HOST: So, it's not a "
            'one-size-fits-all situation? I always tho

# Speech Synthesis Markup Language

In [ ]:
import re

def split_by_speaker(script: str):
    """
    Splits a script where HOST: and CO-HOST: appear inline
    into ordered (speaker, text) tuples.
    """

    pattern = r"(HOST:|CO-HOST:)"
    parts = re.split(pattern, script)

    speaker_lines = []
    current_speaker = None

    for part in parts:
        part = part.strip()
        if not part:
            continue

        if part in ("HOST:", "CO-HOST:"):
            current_speaker = part[:-1]  # remove colon
        else:
            if current_speaker:
                speaker_lines.append(
                    (current_speaker, part.strip())
                )

    return speaker_lines


In [ ]:
print(podcast_scripts[0]["script"])
speaker_lines = split_by_speaker(podcast_scripts[0]["script"])
print(speaker_lines)


HOST: Welcome back to the show! Today, we're diving deep into a topic that's been heating up the tech world: the battle of the AI models. Ever since OpenAI launched ChatGPT in late 2022, we’ve seen a massive race among tech giants to develop and refine their own AI tools. CO-HOST: It really feels like a new AI assistant pops up every other week! It can be overwhelming trying to figure out which one to even try, let alone which one is "the best." HOST: You've hit on the core challenge, because as it turns out, there isn't one single "best" model. What's clear from recent comparisons is that the ideal AI model really depends on what you're trying to achieve. We're talking about ChatGPT, Claude, and Gemini as the major players here. CO-HOST: So, it's not a one-size-fits-all situation? I always thought these AI models were pretty similar across the board. What are some of the key differences we’re seeing? HOST: Let's start with coding, a big one for many of our listeners. If you're looking

In [ ]:
VOICE_MAP = {
    "HOST": "en_US-amy-medium",
    "CO-HOST": "en_US-lessac-medium",
}


In [ ]:
def build_ssml(speaker_lines):
    ssml = "<speak>\n"

    for speaker, text in speaker_lines:
        voice = VOICE_MAP.get(speaker)
        ssml += f"""
        <voice name="{voice}">
            <break time="300ms"/>
            {text}
        </voice>
        """

    ssml += "\n</speak>"
    return ssml


# 6. Text to speech.


In [ ]:
%pip install piper-tts
%pip install pydub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.1 MB/s eta 0:00:00


In [ ]:
!python3 -m piper.download_voices

ar_JO-kareem-low
ar_JO-kareem-medium
bg_BG-dimitar-medium
ca_ES-upc_ona-medium
ca_ES-upc_ona-x_low
ca_ES-upc_pau-x_low
cs_CZ-jirka-low
cs_CZ-jirka-medium
cy_GB-bu_tts-medium
cy_GB-gwryw_gogleddol-medium
da_DK-talesyntese-medium
de_DE-eva_k-x_low
de_DE-karlsson-low
de_DE-kerstin-low
de_DE-mls-medium
de_DE-pavoque-low
de_DE-ramona-low
de_DE-thorsten-high
de_DE-thorsten-low
de_DE-thorsten-medium
de_DE-thorsten_emotional-medium
el_GR-rapunzelina-low
el_GR-rapunzelina-medium
en_GB-alan-low
en_GB-alan-medium
en_GB-alba-medium
en_GB-aru-medium
en_GB-cori-high
en_GB-cori-medium
en_GB-jenny_dioco-medium
en_GB-northern_english_male-medium
en_GB-semaine-medium
en_GB-southern_english_female-low
en_GB-vctk-medium
en_US-amy-low
en_US-amy-medium
en_US-arctic-medium
en_US-bryce-medium
en_US-danny-low
en_US-hfc_female-medium
en_US-hfc_male-medium
en_US-joe-medium
en_US-john-medium
en_US-kathleen-low
en_US-kristin-medium
en_US-kusal-medium
en_US-l2arctic-medium
en_US-lessac-high
en_US-lessac-low
en_US-l

In [ ]:
!python3 -m piper.download_voices en_US-amy-medium

!python3 -m piper.download_voices en_US-lessac-medium


INFO:__main__:Downloaded: en_US-amy-medium
INFO:__main__:Downloaded: en_US-lessac-medium


In [ ]:
import subprocess, shlex

def piper_tts(text, voice, out_wav):
    cmd = f'python3 -m piper -m {voice} -f {out_wav} -- {shlex.quote(text)}'
    subprocess.run(cmd, shell=True, check=True)


In [ ]:
def generate_segment_audio(segment):
    speaker_lines = split_by_speaker(segment["script"])
    wavs = []

    for i, (speaker, text) in enumerate(speaker_lines):
        wav_file = f"seg{segment['segment_id']}_line{i:03d}.wav"
        piper_tts(text, VOICE_MAP[speaker], wav_file)
        wavs.append(wav_file)

    return wavs


In [ ]:
from pydub import AudioSegment

def merge_wavs(wavs, out_file, pause_ms=300):
    final = AudioSegment.empty()
    for w in wavs:
        final += AudioSegment.from_wav(w)
        final += AudioSegment.silent(duration=pause_ms)
    final.export(out_file, format="wav")


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
wavs = generate_segment_audio(segment)
merge_wavs(wavs, f"segment_{segment['segment_id']}.wav")


In [ ]:
def script_to_clips(script):
    clips = []
    speaker_lines = split_by_speaker(script)
    for i, (speaker, text) in enumerate(speaker_lines):
        out = f"clip_{i:03d}.wav"
        piper_tts(text, VOICE_MAP[speaker], out)
        clips.append(out)
    return clips


In [ ]:
from pydub import AudioSegment

def merge_clips(wavs, out_file, pause_ms=300):
    final = AudioSegment.empty()
    for w in wavs:
        final += AudioSegment.from_wav(w)
        final += AudioSegment.silent(duration=pause_ms)
    final.export(out_file, format="wav")


In [ ]:
segment = podcast_scripts[0]  # first segment
clips = script_to_clips(segment["script"])
merge_clips(clips, "segment_11.wav")


In [ ]:
import subprocess
import tempfile
import os

def tts_with_piper(text, voice_path, output_wav):
    process = subprocess.Popen(
        [
            "piper",
            "--model", voice_path,
            "--output_file", output_wav
        ],
        stdin=subprocess.PIPE
    )
    process.communicate(text.encode("utf-8"))


In [ ]:
def generate_audio_clips(script, voice_map, voice_dir):
    clips = []

    lines = script.split("\n")
    clip_id = 0

    for line in lines:
        line = line.strip()
        if not line:
            continue

        if line.startswith("HOST:"):
            speaker = "HOST"
            text = line.replace("HOST:", "").strip()
        elif line.startswith("CO-HOST:"):
            speaker = "CO-HOST"
            text = line.replace("CO-HOST:", "").strip()
        else:
            continue

        voice_file = os.path.join(voice_dir, voice_map[speaker])
        output_wav = f"clip_{clip_id}.wav"

        tts_with_piper(text, voice_file, output_wav)
        clips.append(output_wav)

        clip_id += 1

    return clips


In [ ]:
from pydub import AudioSegment

def merge_wavs(wav_files, output_file):
    final_audio = AudioSegment.empty()

    for wav in wav_files:
        audio = AudioSegment.from_wav(wav)
        final_audio += audio + AudioSegment.silent(duration=300)

    final_audio.export(output_file, format="wav")


In [ ]:
clips = generate_audio_clips(
    script=segment["script"],
    voice_map=VOICE_MAP,
    voice_dir="~/piper-voices"
)

merge_wavs(clips, "segment_1.wav")


FileNotFoundError: [Errno 2] No such file or directory: 'clip_0.wav'

# rough work area


In [ ]:
# for segment in plan:
#     docs = retrieve_module.search(segment["queries"])
#     facts = verify_module.verify_passages(docs, segment)
#     script = script_gen.generate(segment, facts)
#     final_script.append(script)

In [ ]:
%pip install faiss-cpu
%pip install newsapi-python
%pip install sentence-transformers
%pip install transformers accelerate sentencepiece
%pip install transformers accelerate sentencepiece bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 83.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstallin

In [ ]:
import requests
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from google.colab import userdata


# Your NewsAPI key (sign up on https://newsapi.org)
Newsapi_API_KEY = userdata.get('Newsapi_API_KEY')
QUERY = "top news today"

# 1. Get news articles
url = f"https://newsapi.org/v2/everything?q={QUERY}&language=en&sortBy=publishedAt&apiKey={Newsapi_API_KEY}"
response = requests.get(url)
articles = response.json()['articles']
documents = [article['title'] + " - " + article['description'] for article in articles[:20]]

# 2. Create embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(documents)

# 3. Create FAISS index and search
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings))

query_embedding = model.encode([QUERY])
distances, indices = index.search(query_embedding, k=20)

# 4. Print top results
print("Top 5 Internet Results:\n")
for rank, idx in enumerate(indices[0]):
    print(f"{rank+1}: {documents[idx]} (Score: {distances[0][rank]:.4f})")


In [ ]:
%pip install TTS


ERROR: Ignored the following versions that require a different python version: 0.0.10.2 Requires-Python >=3.6.0, <3.9; 0.0.10.3 Requires-Python >=3.6.0, <3.9; 0.0.11 Requires-Python >=3.6.0, <3.9; 0.0.12 Requires-Python >=3.6.0, <3.9; 0.0.13.1 Requires-Python >=3.6.0, <3.9; 0.0.13.2 Requires-Python >=3.6.0, <3.9; 0.0.14.1 Requires-Python >=3.6.0, <3.9; 0.0.15 Requires-Python >=3.6.0, <3.9; 0.0.15.1 Requires-Python >=3.6.0, <3.9; 0.0.9 Requires-Python >=3.6.0, <3.9; 0.0.9.1 Requires-Python >=3.6.0, <3.9; 0.0.9.2 Requires-Python >=3.6.0, <3.9; 0.0.9a10 Requires-Python >=3.6.0, <3.9; 0.0.9a9 Requires-Python >=3.6.0, <3.9; 0.1.0 Requires-Python >=3.6.0, <3.10; 0.1.1 Requires-Python >=3.6.0, <3.10; 0.1.2 Requires-Python >=3.6.0, <3.10; 0.1.3 Requires-Python >=3.6.0, <3.10; 0.10.0 Requires-Python >=3.7.0, <3.11; 0.10.1 Requires-Python >=3.7.0, <3.11; 0.10.2 Requires-Python >=3.7.0, <3.11; 0.11.0 Requires-Python >=3.7.0, <3.11; 0.11.1 Requires-Python >=3.7.0, <3.11; 0.12.0 Requires-Python >=3

In [ ]:
!tts \
  --text "Hello from Kerala!" \
  --model_name tts_models/multilingual/multi-dataset/your_tts \
  --speaker_idx "male-en-2" \
  --language_idx "en"

/bin/bash: line 1: tts: command not found


In [ ]:
!tts --model_name tts_models/multilingual/multi-dataset/your_tts --list_speaker_idxs


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_name = "meta-llama/Meta-Llama-3-8B-Instruct"

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load the model with 4-bit quantization (to save memory)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True
)

# Create a pipeline for text generation
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)


In [ ]:
prompt = "Write a podcast script about the latest space exploration news."
result = pipe(prompt, max_new_tokens=300, do_sample=True, temperature=0.7)
print(result[0]['generated_text'])


In [ ]:
import serpapi
print(dir(serpapi))

['APIKeyNotProvided', 'Client', 'HTTPClient', 'HTTPConnectionError', 'HTTPError', 'SearchIDNotProvided', 'SerpApiError', 'SerpResults', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'account', 'core', 'exceptions', 'http', 'locations', 'models', 'requests', 'search', 'search_archive', 'textui']


In [ ]:
%pip install faiss-cpu

  Using cached faiss_cpu-1.11.0.post1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (5.0 kB)
Using cached faiss_cpu-1.11.0.post1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (31.3 MB)


In [ ]:
%pip install -U langchain
# Requires Python 3.10+

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.1.0
    Uninstalling langchain-1.1.0:
      Successfully uninstalled langchain-1.1.0


In [ ]:
%pip install -U langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 3.6 MB/s eta 0:00:00


In [ ]:
import google.generativeai as genai
from google.colab import userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

# Configure Gemini API key
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# Initialize the Gemini model with LangChain
# You can specify the model name, for example, 'gemini-pro'
llm = ChatGoogleGenerativeAI(model="gemini-pro")

# Example usage:
message = HumanMessage(content="What are the benefits of using AI in education?")
response = llm.invoke([message])

print(response.content)

First, let's install the necessary libraries. We'll use `langchain-community` for document loaders, `faiss-cpu` for the vector store, `sentence-transformers` for embeddings, and `langchain-openai` for the LLM (you can replace this with `langchain-google-genai` if you prefer to use Gemini). We'll also need `google-generativeai` and `openai` for the LLMs.

In [ ]:
pip install -U langchain langchain-community faiss-cpu sentence-transformers openai

Now, let's set up our environment and create some dummy documents for demonstration. In a real-world scenario, you would load documents from files, databases, or web pages.

In [ ]:
from langchain.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain.embeddings import SentenceTransformerEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.chains import RetrievalQA
from langchain_openai import OpenAI
import os
from google.colab import userdata

# Set your API key for OpenAI (or use GOOGLE_API_KEY if using Gemini for LLM)
# You can also use userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') # Replace with your actual OpenAI API key

# 1. Load Documents
# For simplicity, we'll create a dummy text file. In practice, you'd load from your data source.
with open('rag_document.txt', 'w') as f:
    f.write("""
LangChain is a framework designed to simplify the creation of applications using large language models (LLMs). It provides tools for building powerful applications like chatbots, summarization tools, and question-answering systems over specific knowledge bases (RAG).

Retrieval-Augmented Generation (RAG) is an AI framework that retrieves factual information from external knowledge bases and then uses that information to generate more accurate and informative responses. It helps LLMs overcome limitations like knowledge cutoff and hallucination.

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search and clustering of dense vectors. It's often used as a vector store in RAG pipelines.

Sentence-Transformers is a Python framework for state-of-the-art sentence, text and image embeddings. It can be used to compute dense vector representations for sentences, paragraphs, and images.
""")

loader = TextLoader('rag_document.txt')
documents = loader.load()

# 2. Split Documents into Chunks
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=0)
texts = text_splitter.split_documents(documents)


Next, we'll create embeddings for our document chunks and store them in a FAISS vector database. This allows for efficient similarity search.

In [ ]:
# 3. Create Embeddings and Vector Store
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")
db = FAISS.from_documents(texts, embeddings)


Finally, we'll set up a RetrievalQA chain, which combines a retriever (our FAISS vector store) with an LLM to answer questions based on the retrieved context.

In [ ]:
# 4. Set up the RAG Chain
llm = OpenAI(temperature=0) # You can replace with ChatGoogleGenerativeAI(model="gemini-pro")
qa_chain = RetrievalQA.from_chain_type(llm, retriever=db.as_retriever())

# 5. Perform a RAG Query
query = "What is RAG and why is it useful?"
response = qa_chain.invoke({"query": query})

print("Query:", query)
print("Response:", response["result"])